In [ ]:
import read_eps
import  matplotlib.pyplot as plt
import numpy as np
import lab
import potcorr


import const
import h5py
import pickle

In [ ]:
import matplotlib as mpl

from scipy.interpolate import interp1d

# 生成一些数据
data = np.random.randn(30, 30)

# 选择一个预设的colormap
original_cmap = plt.cm.PuOr_r

# 定义新旧colormap之间的非线性映射
new_positions = [0, 0.5, 1]  # 新colormap中的位置
old_positions = [0, 0.5, 0.75]  # 旧colormap中对应的位置
mapping_function = interp1d(new_positions, old_positions)

# 使用映射函数生成自定义colormap的颜色列表
custom_colors = original_cmap(mapping_function(np.linspace(0, 1, 256)))

# 创建自定义colormap
custom_cmap = mpl.colors.LinearSegmentedColormap.from_list('custom_cmap', custom_colors)

# 创建图表并应用自定义colormap
plt.imshow(data, cmap=custom_cmap)
plt.colorbar()
plt.show()

In [ ]:
cell=lab.mos2_unit_tt
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()


E_f  = 2.87293035e-02 # eV
eff_m = 0.499 # electron mass

eps1='/anvil/projects/x-che190065/rjguo/mos2/dielectric/chi_for_zz/12x12_cut30/chimat.h5'
eps0='/anvil/projects/x-che190065/rjguo/mos2/dielectric/chi_for_zz/12x12_cut30/chi0mat.h5'
ttkw.read_epsinv(eps1=eps1, eps0=eps0)

In [ ]:
Eps0 = ttkw.Eps0
Eps1 = ttkw.Eps1

for gg in range(0,1):
    g0_ = [0,0,0]
    g1_ = [0,0,0]
    q0_ = Eps0.qpts[:]
    q1_ = Eps1.qpts[:]
    q0_abs_ = np.sqrt((q0_[:,0])**2+ (np.sqrt(3)/3*q0_[:,0]+2*np.sqrt(3)/3*q0_[:,1])**2)
    print(q0_abs_)
    q1_abs_ = np.sqrt((q1_[:,0])**2+ (np.sqrt(3)/3*q1_[:,0]+2*np.sqrt(3)/3*q1_[:,1])**2)
    eps_Re_ = []
    eps_Im_ = []
    wcoul_Re_ = []
    wcoul_Im_ = []
    q_abs_ = []
    for i in range(10):
        qind_ = i
        g_vec0_ = tuple(g0_)
        g_vec1_ = tuple(g1_)
        gind_rho0_ = Eps0.G_vec2ind[g_vec0_]
        gind_rho1_ = Eps0.G_vec2ind[g_vec1_]
        gind_eps0_ = Eps0.gind_rho2eps[qind_, gind_rho0_]
        gind_eps1_ = Eps0.gind_rho2eps[qind_, gind_rho1_]
        
        mat_Re_ = Eps0.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,0]
        mat_Im_ = Eps0.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,1]
        eps_Re_.append(mat_Re_)
        eps_Im_.append(mat_Im_)
        q_abs_of_ = np.sqrt((q0_[i,0])**2+ (np.sqrt(3)/3*q0_[i,0]+2*np.sqrt(3)/3*q0_[i,1])**2)
        q_abs_ = np.hstack((q_abs_, q_abs_of_))
        
    for i in range(0,30):
        qind_ = i
        g_vec0_ = tuple(g0_)
        g_vec1_ = tuple(g1_)
        gind_rho0_ = Eps1.G_vec2ind[g_vec0_]
        gind_rho1_ = Eps1.G_vec2ind[g_vec1_]
        gind_eps0_ = Eps1.gind_rho2eps[qind_, gind_rho0_]
        gind_eps1_ = Eps1.gind_rho2eps[qind_, gind_rho1_]
        
        mat_Re_ = Eps1.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,0]
        mat_Im_ = Eps1.mat[qind_,0,0,gind_eps1_-1,gind_eps0_-1,1]
        eps_Re_.append(mat_Re_)
        eps_Im_.append(mat_Im_)
        
        q_abs_of_ = np.sqrt((q1_[i,0])**2+ (np.sqrt(3)/3*q1_[i,0]+2*np.sqrt(3)/3*q1_[i,1])**2)
        q_abs_ = np.hstack((q_abs_, q_abs_of_))
        
        #mat_Re_ = ttkw.wcoul[qind_,gind_eps0_-1,gind_eps1_-1].real
        #mat_Im_ = ttkw.wcoul[qind_,gind_eps0_-1,gind_eps1_-1].imag
        #wcoul_Re_.append(mat_Re_)
        #wcoul_Im_.append(mat_Im_)
    #q_abs_ = np.hstack((q0_abs_, q1_abs_ ))
    
    fig, ax = plt.subplots(figsize = (12,6), dpi = 100)
    ax.scatter(q_abs_, np.array(eps_Re_))
    ax.scatter(q_abs_, eps_Im_)
    # ax.scatter(q_abs_, eps_re)
    #ax.scatter(q_abs_[:10], eps_Re_[:10])
    #ax.plot(q_abs_[:20], eps_Re_[:20])
    #plt.xlim(-0.01,0.02)
    #plt.ylim(-0.01,0.01)
    plt.show()

In [ ]:
def BGW2obz(Chi=ttkw.Eps1, qind=1,Gy_= 0, nGz=71):
    nmtx_ = Chi.nmtx
    qind_ = qind
    #print(ttkw.Eps1.qlist[qind_])
    gin_eps_ = ttkw.Eps1.gind_eps2rho[qind_][:nmtx_[qind_]]

    Gz_list = np.fft.fftfreq(nGz,d=1/nGz)
    Gz_list2 = -np.fft.fftfreq(nGz,d=1/nGz)
    #print(Gz_list[-35:])

    Gind_list = []
    Gind_list2 = []
    for i in Gz_list:
        G_vec_ = tuple((0,Gy_,i))
        G_ind_rho = Chi.G_vec2ind[G_vec_]
        G_ind_eps = Chi.gind_rho2eps[qind_][G_ind_rho]
        #print(G_vec_)
        #print(G_ind_)
        Gind_list.append(G_ind_eps)

    for i in Gz_list2:
        G_vec_ = tuple((0,Gy_,i))
        G_ind_rho = Chi.G_vec2ind[G_vec_]
        G_ind_eps = Chi.gind_rho2eps[qind_][G_ind_rho]
        #print(G_vec_)
        #print(G_ind_)
        Gind_list2.append(G_ind_eps)

    chi_GzGz=np.zeros([len(Gind_list),len(Gind_list)], dtype=complex)
    for i,g0 in enumerate(Gind_list):
        for j,g1 in enumerate(Gind_list2):
            chi_real = Chi.mat[qind_,0,0,g1-1,g0-1,0]
            chi_imag = Chi.mat[qind_,0,0,g1-1,g0-1,1]
            chi_GzGz[i,j]=chi_real+1j*chi_imag
    return chi_GzGz

def get_kf(E_f, eff_m):
    return np.sqrt(2*eff_m*E_f/2/const.Ry2eV) # unit: bohr^(-1) 

def get_chi_2DEG(q, kf, eff_m, Ns, Nv):
    return np.where(q>2*kf, -Ns*Nv*eff_m/4/np.pi*(1-np.sqrt(1-4*kf**2/q**2)),-Ns*Nv*eff_m/4/np.pi ) # energy unit: Ry

#E_f  = 2.11408387e-03#test2
#E_f  = 8.11408387e-04 #conc3e12

E_f = 2.82494709e-8 # conc1e12
#E_f  = 2.87293035e-02
#E_f  = 8.87293035e-02
#E_f = 0.1
#E_f  =2.82494709e-05
kf = get_kf(E_f, eff_m)#*const.Bohr_R
print(kf)
kf = 0.07

chi_rbf_path = '/anvil/projects/x-che190065/rjguo/mos2/dielectric/adler-wiser'
with open(chi_rbf_path+'/chi_300k_3e13_rbf.pkl', 'rb') as file:
    chi_3e13_rbf = pickle.load(file)

with open(chi_rbf_path+'/chi_300k_1e13_rbf.pkl', 'rb') as file:
    chi_1e13_rbf = pickle.load(file)

with open(chi_rbf_path+'/chi_300k_3e12_rbf.pkl', 'rb') as file:
    chi_3e12_rbf = pickle.load(file)

with open(chi_rbf_path+'/chi_300k_1e12_rbf.pkl', 'rb') as file:
    chi_1e12_rbf = pickle.load(file)
    
    
with open(chi_rbf_path+'/chi_300k_3e11_rbf.pkl', 'rb') as file:
    chi_3e11_rbf = pickle.load(file)
    
with open(chi_rbf_path+'/chi_300k_1e11_rbf.pkl', 'rb') as file:
    chi_1e11_rbf = pickle.load(file)
    
with open(chi_rbf_path+'/chi_300k_3e10_rbf.pkl', 'rb') as file:
    chi_3e10_rbf = pickle.load(file)

with open(chi_rbf_path+'/chi_300k_1e10_rbf.pkl', 'rb') as file:
    chi_1e10_rbf = pickle.load(file)
    
    
chi_rbf = chi_3e13_rbf

In [ ]:
nGz_l = 450

z = np.linspace(-12,12,nGz_l)/const.Bohr_R
dist_zz = np.abs(z[:, np.newaxis] - z[np.newaxis, :])


plt.imshow(dist_zz, cmap='hot', interpolation='nearest')
plt.colorbar()  # 显示颜色条
plt.title("Heatmap of dist(z, z')")
plt.xlabel("z'")
plt.ylabel("z")
plt.gca().invert_yaxis()
#plt.xlim(225//2-20,225//2+20)
#plt.ylim(225//2-20,225//2+20)
plt.show()

In [ ]:
nGz_l = 450

xi_Gz = np.load('/anvil/projects/x-che190065/rjguo/mos2/dielectric/xi/xi_Gz.npy')

xi_Gz_l = np.zeros(nGz_l, dtype=complex)
xi_Gz_l[:225//2] = xi_Gz[:225//2]
xi_Gz_l[-225//2:] = xi_Gz[-225//2:]
xi = np.fft.ifftn(xi_Gz_l)/24*const.Bohr_R*nGz_l
#xi2 = np.zeros_like(xi)
#xi2[:225//2] = xi[225//2+1:]
#xi2[225//2:] = xi[:225//2+1]
#plt.plot(xi.real)
#xi_Gz2 = np.fft.fftn(xi2)

xizz = np.outer(xi.real,xi.real)
xizz_0 = np.zeros_like(xizz) 
xizz_0[225//2,225//2]=1



In [ ]:
plt.plot(xi_Gz_l)

In [ ]:
np.sum(xi)*24/const.Bohr_R/450

In [ ]:
plt.plot(xi)

In [ ]:

nGz=61
nGz_l = 450



epsf= ttkw.Eps0
wlist = []
qabs_list = []
for Gy in range(1):
    for qind in range(1):

        q1_ = epsf.qpts[:]
        q_abs_ = np.sqrt((q1_[qind,0])**2+ (np.sqrt(3)/3*(q1_[qind,0])+2*np.sqrt(3)/3*(q1_[qind,1]+Gy))**2)*2*np.pi/ttkw.lattpara_unit[0]*const.Bohr_R
        vcoul_2d= 2*np.pi * np.exp(-q_abs_*dist_zz) / q_abs_
       # print(q_abs_)
        if q_abs_ < 3.6:
            chi_GzGz = BGW2obz(Chi=epsf, qind=qind, Gy_=Gy, nGz=nGz)

            chi_GzGz_l = np.zeros([nGz_l,nGz_l], dtype=complex)
            chi_GzGz_l[:nGz//2+1,:nGz//2+1]=chi_GzGz[:nGz//2+1,:nGz//2+1]
            chi_GzGz_l[:nGz//2+1,-nGz//2+1:]=chi_GzGz[:nGz//2+1,-nGz//2+1:]
            chi_GzGz_l[-nGz//2+1:,:nGz//2+1]=chi_GzGz[-nGz//2+1:,:nGz//2+1]
            chi_GzGz_l[-nGz//2+1:,-nGz//2+1:]=chi_GzGz[-nGz//2+1:,-nGz//2+1:]

          #  chi_GzGz_s = np.zeros_like(chi_GzGz)
          #  chi_GzGz_s[:nGz//2+1,:nGz//2+1]=chi_GzGz[:nGz//2+1,:nGz//2+1]
          #  chi_GzGz_s[:nGz//2+1,-nGz//2+1:]=chi_GzGz[:nGz//2+1,-nGz//2+1:]
          #  chi_GzGz_s[-nGz//2+1:,:nGz//2+1]=chi_GzGz[-nGz//2+1:,:nGz//2+1]
          #  chi_GzGz_s[-nGz//2+1:,-nGz//2+1:]=chi_GzGz[-nGz//2+1:,-nGz//2+1:]

           # chi_zz_s=np.fft.ifftn(chi_GzGz_s)*71*71/(24/const.Bohr_R)**2
            chi_zz_l=np.fft.ifftn(chi_GzGz_l)*nGz_l*nGz_l/(24/const.Bohr_R)**2


            chi_zz_q2DEG = float(get_chi_2DEG(q_abs_, kf, eff_m, 1, 1)) * xizz
            chi_zz_ftq2DEG = float(0.5*chi_rbf([[q_abs_*const.Bohr_R]])) * xizz
            chi_zz_t =  chi_zz_ftq2DEG+chi_zz_l
            
            chi_zz_intrinsic_plot = chi_zz_l
            

            
            epsilon = np.eye(nGz_l) - 2*np.pi * vcoul_2d@chi_zz_t#*((24/const.Bohr_R)/225)
        else:
            epsilon = np.eye(nGz_l)
        epsilon_inv = np.linalg.inv(epsilon)
        w = epsilon_inv@vcoul_2d#*((24/const.Bohr_R)/225)
        wlist.append(w)
        qabs_list.append(q_abs_)
        '''
        plt.imshow(w.real, cmap='hot', interpolation='nearest')
        plt.colorbar()  # 显示颜色条
        plt.title("Heatmap of chi_qxy(z, z')")
        plt.xlabel("z'")
        plt.ylabel("z")
        plt.gca().invert_yaxis()
        #plt.xlim(225//2-20,225//2+20)
        #plt.ylim(225//2-20,225//2+20)
        plt.show()
        '''
for Gy in range(0):
    epsf= ttkw.Eps1
    for qind in range(1,12):

        q1_ = epsf.qpts[:]
        #print(q1_)
        q_abs_ = np.sqrt((q1_[qind,0])**2+ (np.sqrt(3)/3*(q1_[qind,0])+2*np.sqrt(3)/3*(q1_[qind,1]+Gy))**2)*2*np.pi/ttkw.lattpara_unit[0]*const.Bohr_R

        vcoul_2d= 2*np.pi * np.exp(-q_abs_*dist_zz) / q_abs_
      #  print(q_abs_)
        
        if q_abs_ < 3.6:
            chi_GzGz = BGW2obz(Chi=epsf, qind=qind, Gy_=Gy, nGz=nGz)

            chi_GzGz_l = np.zeros([nGz_l,nGz_l], dtype=complex)
            chi_GzGz_l[:nGz//2+1,:nGz//2+1]=chi_GzGz[:nGz//2+1,:nGz//2+1]
            chi_GzGz_l[:nGz//2+1,-nGz//2+1:]=chi_GzGz[:nGz//2+1,-nGz//2+1:]
            chi_GzGz_l[-nGz//2+1:,:nGz//2+1]=chi_GzGz[-nGz//2+1:,:nGz//2+1]
            chi_GzGz_l[-nGz//2+1:,-nGz//2+1:]=chi_GzGz[-nGz//2+1:,-nGz//2+1:]

        #    chi_GzGz_s = np.zeros_like(chi_GzGz)
        #    chi_GzGz_s[:nGz//2+1,:nGz//2+1]=chi_GzGz[:nGz//2+1,:nGz//2+1]
        #    chi_GzGz_s[:nGz//2+1,-nGz//2+1:]=chi_GzGz[:nGz//2+1,-nGz//2+1:]
        #    chi_GzGz_s[-nGz//2+1:,:nGz//2+1]=chi_GzGz[-nGz//2+1:,:nGz//2+1]
        #    chi_GzGz_s[-nGz//2+1:,-nGz//2+1:]=chi_GzGz[-nGz//2+1:,-nGz//2+1:]

        #    chi_zz_s=np.fft.ifftn(chi_GzGz_s)*71*71/(24/const.Bohr_R)**2
            chi_zz_l=np.fft.ifftn(chi_GzGz_l)*nGz_l*nGz_l/(24/const.Bohr_R)**2


            chi_zz_q2DEG = float(get_chi_2DEG(q_abs_, kf, eff_m, 1, 1)) * xizz
            chi_zz_ftq2DEG = float(0.5*chi_rbf([[q_abs_*const.Bohr_R]])) * xizz
            chi_zz_t = chi_zz_ftq2DEG+chi_zz_l
           # chi_zz_t = chi_zz_q2DEG#+chi_zz_l
        
            epsilon = np.eye(nGz_l) - 2*np.pi * vcoul_2d@chi_zz_t#*((24/const.Bohr_R)/225)
        else:
            epsilon = np.eye(nGz_l)
        epsilon_inv = np.linalg.inv(epsilon)
        w = epsilon_inv@vcoul_2d#*((24/const.Bohr_R)/225)
        wlist.append(w)
        qabs_list.append(q_abs_)

In [ ]:
from matplotlib.ticker import ScalarFormatter
fig, ax = plt.subplots(figsize = (2,2), dpi = 600)
plt.imshow(chi_zz_t.real, cmap=custom_cmap, interpolation='nearest',extent=[-12, 12, 12, -12], vmax=0.0001, vmin = -0.01)
#cbar = plt.colorbar()  # 显示颜色条
#plt.title("Heatmap of chi_qxy(z, z')")
plt.xlabel("z ($\AA$)")
plt.ylabel("z' ($\AA$)")
plt.gca().invert_yaxis()
plt.xlim(-7,7)
plt.ylim(-7,7)

formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((-1, 1))
cbar.ax.yaxis.set_major_formatter(formatter)
plt.show()

In [ ]:

qabs_list = qabs_list
wlist = np.array(wlist)

plt.scatter(qabs_list, wlist[:,225//2,225//2].real, label='intrinsic')
plt.plot(qabs_list, wlist[:,225//2,225//2].real, linestyle='--')

#plt.xlim(0,0.1)

In [ ]:

plt.scatter(qabs_list5, wlist5[:,225//2,225//2].real, label='1e11')
plt.plot(qabs_list5, wlist5[:,225//2,225//2].real, linestyle='--')

plt.scatter(qabs_list4, wlist4[:,225//2,225//2].real, label='1e12')
plt.plot(qabs_list4, wlist4[:,225//2,225//2].real, linestyle='--')

plt.scatter(qabs_list1, wlist1[:,225//2,225//2].real, label='1e13')
plt.plot(qabs_list1, wlist1[:,225//2,225//2].real, linestyle='--')

plt.scatter(qabs_list2, wlist2[:,225//2,225//2].real, label='1e14')
plt.plot(qabs_list2, wlist2[:,225//2,225//2].real, linestyle='--')

plt.scatter(qabs_list3, wlist3[:,225//2,225//2].real, label='3e14')
plt.plot(qabs_list3, wlist3[:,225//2,225//2].real, linestyle='--')



plt.xlim(-0.05,0.5)
plt.ylim(-1,15)

plt.legend()
plt.show()

In [ ]:
from scipy.interpolate import interp1d
w_zz_interp = []
for i in range(225):
    print(i)
    w_z_interp = []
    for j in range(225):
        linear_interp = interp1d(np.array(qabs_list),np.array(wlist)[:,i,j])

        #q_ = np.linspace(1e-3,24,10000)
        #w_ = linear_interp(q_)
        w_z_interp.append(linear_interp)
    w_zz_interp.append(w_z_interp)

       # plt.scatter(qabs_list, np.array(wlist)[:,0,0].real)
       # plt.scatter(q_,w_, s=2)
       # plt.xlim(-1,10)

In [ ]:
q_ = np.linspace(1e-3,24,10000)

In [ ]:
plt.scatter(np.array(qabs_list),np.array(wlist)[:,225//2,225//2].real, label = 'q2DEG+intrinsic')
plt.scatter(q_,w_zz_interp[225//2][225//2](q_).real, s=3, label = 'interpolated')
plt.legend()
plt.xlim(-0.1,0.5)
#plt.ylim(-0.01,0.02)

In [ ]:
plt.imshow(chi_GzGz.real, cmap='hot', interpolation='nearest')
plt.colorbar()  # 显示颜色条
plt.title("Heatmap of chi_qxy(Gz, Gz')")
plt.xlabel("Gz'")
plt.ylabel("Gz")
plt.gca().invert_yaxis()
#plt.xlim(225//2-20,225//2+20)
#plt.ylim(225//2-20,225//2+20)
plt.show()

plt.imshow(chi_GzGz_l.real, cmap='hot', interpolation='nearest')
plt.colorbar()  # 显示颜色条
plt.title("Heatmap of chi_qxy(Gz, Gz')")
plt.xlabel("Gz'")
plt.ylabel("Gz")
plt.gca().invert_yaxis()
#plt.xlim(0,20)
#plt.ylim(0,20)
plt.show()

plt.imshow(chi_zz_s.real, cmap='hot', interpolation='nearest')
plt.colorbar()  # 显示颜色条
plt.title("Heatmap of chi_qxy(z, z')")
plt.xlabel("z'")
plt.ylabel("z")
plt.gca().invert_yaxis()
#plt.xlim(225//2-20,225//2+20)
#plt.ylim(225//2-20,225//2+20)
plt.show()

plt.imshow(chi_zz_l.real, cmap='hot', interpolation='nearest')
plt.colorbar()  # 显示颜色条
plt.title("Heatmap of chi_qxy(z, z')")
plt.xlabel("z'")
plt.ylabel("z")
plt.gca().invert_yaxis()
#plt.xlim(225//2-20,225//2+20)
#plt.ylim(225//2-20,225//2+20)
plt.show()

In [ ]:
#from matplotlib.ticker import FuncFormatter, MultipleLocator
#interval = 225/13

#def format_ticks(value, pos):
    # 将输入范围[0, 9]映射到[-1, 1]
#    return '{:.1f}'.format((value)/ interval - 12)

fig, ax = plt.subplots(figsize = (12,6), dpi = 600)
plt.imshow(chi_zz_t.real, cmap='hot', interpolation='nearest',extent=[-12, 12, -12, 12], vmax=0.001, vmin = -0.01)
plt.colorbar()  # 显示颜色条
plt.title("Heatmap of chi_qxy(z, z')")
plt.xlabel("z ($\AA$)")
plt.ylabel("z' ($\AA$)")
plt.gca().invert_yaxis()
#ax.xaxis.set_major_formatter(FuncFormatter(format_ticks))
#ax.yaxis.set_major_formatter(FuncFormatter(format_ticks))
#ax.xaxis.set_major_locator(MultipleLocator(interval))
#ax.yaxis.set_major_locator(MultipleLocator(interval))
#plt.xlim(225//2-20,225//2+20)
#plt.ylim(225//2-20,225//2+20)
plt.show()

In [ ]:
cell=lab.mos2_12to48
ttkw2=potcorr.PotCorr(cell)
ttkw2.fft_init()

In [ ]:
rho_bare=np.load('/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/rho_ext.npy')

nx, ny, nz = rho_bare.shape

#----- Moving the defect center to (0,0)
rho_bare2 = np.zeros_like(rho_bare)
rho_bare2[:nx//2,:nx//2,:]=rho_bare[nx//2:,nx//2:,:]
rho_bare2[:nx//2,-nx//2:,:]=rho_bare[nx//2:,:nx//2,:]
rho_bare2[-nx//2:,:nx//2,:]=rho_bare[:nx//2,-nx//2:,:]
rho_bare2[-nx//2:,-nx//2:,:]=rho_bare[:nx//2,:nx//2,:]

fine_data = rho_bare2
f_transform = np.fft.fftn(fine_data)



new_f_transform = np.zeros((nx//2,ny//2,nz), dtype=complex)
nx, ny, nz = new_f_transform.shape

new_f_transform[:nx//2, :ny//2, :nz//2] = f_transform[:nx//2, :ny//2, :nz//2]
new_f_transform[-nx//2:, :ny//2, :nz//2] = f_transform[-nx//2:, :ny//2, :nz//2]
new_f_transform[:nx//2, -ny//2:, :nz//2] = f_transform[:nx//2, -ny//2:, :nz//2]
new_f_transform[-nx//2:, -ny//2:, :nz//2] = f_transform[-nx//2:, -ny//2:, :nz//2]
new_f_transform[:nx//2, :ny//2, -nz//2:] = f_transform[:nx//2, :ny//2, -nz//2:]
new_f_transform[-nx//2:, :ny//2, -nz//2:] = f_transform[-nx//2:, :ny//2, -nz//2:]
new_f_transform[:nx//2, -ny//2:, -nz//2:] = f_transform[:nx//2, -ny//2:, -nz//2:]
new_f_transform[-nx//2:, -ny//2:, -nz//2:] = f_transform[-nx//2:, -ny//2:, -nz//2:]


fine_data2 = np.fft.ifftn(new_f_transform).real/4

rho_bare_l = np.zeros((720,720,225), dtype=complex )

rho_bare_l[:nx//2, :nx//2, :] = fine_data2[:nx//2, :nx//2, :]
rho_bare_l[:nx//2, -nx//2:, :] = fine_data2[:nx//2, -nx//2:, :]
rho_bare_l[-nx//2:, :nx//2, :] = fine_data2[-nx//2:, :nx//2, :]
rho_bare_l[-nx//2:, -nx//2:, :] = fine_data2[-nx//2:, -nx//2:, :]


In [ ]:
rho_bare2 = np.zeros_like(rho_bare)

rho_bare2[:720//2,:720//2,:]=rho_bare[720//2:,720//2:,:]
rho_bare2[:720//2,-720//2:,:]=rho_bare[720//2:,:720//2,:]
rho_bare2[-720//2:,:720//2,:]=rho_bare[:720//2,-720//2:,:]
rho_bare2[-720//2:,-720//2:,:]=rho_bare[:720//2,:720//2,:]

In [ ]:
fine_data = rho_bare2
f_transform = np.fft.fftn(fine_data)

# 获取原始数据的维度
nx, ny, nz = f_transform.shape

# 创建一个用0填充的更大的傅里叶空间数组
new_f_transform = np.zeros((360,360,225), dtype=complex)
nx, ny, nz = new_f_transform.shape
# 将原始的傅里叶变换数据复制到新的傅里叶空间中心
new_f_transform[:nx//2, :ny//2, :nz//2] = f_transform[:nx//2, :ny//2, :nz//2]
new_f_transform[-nx//2:, :ny//2, :nz//2] = f_transform[-nx//2:, :ny//2, :nz//2]
new_f_transform[:nx//2, -ny//2:, :nz//2] = f_transform[:nx//2, -ny//2:, :nz//2]
new_f_transform[-nx//2:, -ny//2:, :nz//2] = f_transform[-nx//2:, -ny//2:, :nz//2]
new_f_transform[:nx//2, :ny//2, -nz//2:] = f_transform[:nx//2, :ny//2, -nz//2:]
new_f_transform[-nx//2:, :ny//2, -nz//2:] = f_transform[-nx//2:, :ny//2, -nz//2:]
new_f_transform[:nx//2, -ny//2:, -nz//2:] = f_transform[:nx//2, -ny//2:, -nz//2:]
new_f_transform[-nx//2:, -ny//2:, -nz//2:] = f_transform[-nx//2:, -ny//2:, -nz//2:]

# 执行逆傅里叶变换并缩放（因为数组大小已经改变）
fine_data2 = np.fft.ifftn(new_f_transform).real *( 360/720*360/720*225/225)

In [ ]:
rho_bare_l = np.zeros((720,720,225), dtype=complex )

rho_bare_l[:360//2, :360//2, :] = fine_data2[:360//2, :360//2, :]
rho_bare_l[:360//2, -360//2:, :] = fine_data2[:360//2, -360//2:, :]
rho_bare_l[-360//2:, :360//2, :] = fine_data2[-360//2:, :360//2, :]
rho_bare_l[-360//2:, -360//2:, :] = fine_data2[-360//2:, -360//2:, :]

#rho_bare_l=np.pad(fine_data2,((180,180),(180,180),(0,0)), 'constant', constant_values=(0,0))
#rho_bare_l[-360//2:, :360//2, :] = fine_data2[-360//2:, :360//2, :]
#rho_bare_l[-360//2:, -360//2:, :] = fine_data2[-360//2:, -360//2:, :]

In [ ]:
x_plot = (ttkw2.fft_xx[:,:,ttkw2.fft_nz//2]-0.5*ttkw2.fft_yy[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_yy[:,:,ttkw2.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=rho_bare_l[:,:,225//2].real, s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
rho_bare_l = rho_bare_l*(1/(np.sum(rho_bare_l)*ttkw2.omega/720/720/225))
#rho_bare2 = rho_bare2*(1/(np.sum(rho_bare2)*ttkw2.omega/720/720/225/4))

In [ ]:
np.sum(rho_bare_l)*ttkw2.omega/720/720/225

In [ ]:
fft_q_abs_ = np.sqrt(ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]**2 + (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2])**2)
fft_q_abs_[0,0] = 0.00027
#fft_w_ = linear_interp(fft_q_abs_)

In [ ]:
w_zz_q0_ = np.zeros((225,225))
for i in range(225):
    for j in range(225):
        w_zz_q0_[i,j]=float(w_zz_interp[i][j](2).real)

plt.imshow(w_zz_q0_.real, cmap='hot', interpolation='nearest')
plt.colorbar()  # 显示颜色条
plt.title("Heatmap of w_qxy(z, z')")
plt.xlabel("z'")
plt.ylabel("z")
plt.gca().invert_yaxis()
#plt.xlim(225//2-20,225//2+20)
#plt.ylim(225//2-20,225//2+20)
plt.show()

In [ ]:
ttkw2.lattpara[2]/225

In [ ]:
#w_zz_ = w_zz_interp[225//2][225//2](fft_q_abs_)
x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=fft_q_abs_, s=0.8,
                #gridsize=100,
                vmax=0.05,
                vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
pot_k_z = np.zeros([720,720,225], dtype=complex)
for i in range(225):
    print(i)
    for j in range(225//2-30,225//2+30):
        rho_k_l_ = np.fft.fftn(rho_bare_l[:,:,j])*ttkw2.lattpara[0]*ttkw2.lattpara[1]/720/720*np.sqrt(3)/2
        w_zz_ = w_zz_interp[i][j](fft_q_abs_)
        pot_k_z[:,:,i]= pot_k_z[:,:,i] + w_zz_ * rho_k_l_*ttkw2.lattpara[2]/225

In [ ]:

x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.fft.fftn(rho_bare_l[:,:,225//2+10]).real*ttkw2.lattpara[0]*ttkw2.lattpara[1]/720/720, s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
 plt.plot(np.sum(rho_bare_l, axis=(0,1)))

In [ ]:
x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot_k_z[:,:,225//2+0].real, s=28,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
pot_r = np.zeros_like(pot_k_z)
for i in range(225):
    pot_r[:,:,i] = np.fft.ifftn(pot_k_z[:,:,i]).real

In [ ]:
x_plot = (ttkw2.fft_xx[:,:,ttkw2.fft_nz//2]-0.5*ttkw2.fft_yy[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_yy[:,:,ttkw2.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot_r[:,:,225//2+100], s=1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
#pot_r2 = np.load('/anvil/projects/x-che190065/rjguo/mos2/potential/2d_new/pot_test.npy')
np.save('/anvil/projects/x-che190065/rjguo/mos2/potential/2d_new/pot_k_0.07_2.npy',pot_k_z[:,:,225//2-30: 225//2+30] )

In [ ]:
plt.plot(pot_r[720//2,:,225//2])

In [ ]:
rho_k = np.fft.fftn(rho_bare2[:,:,225//2])*ttkw2.lattpara[0]*ttkw2.lattpara[1]/720/720/4
rho_k_l = np.fft.fftn(rho_bare_l[:,:,225//2])*ttkw2.lattpara[0]*ttkw2.lattpara[1]/720/720

In [ ]:
x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=rho_k[:,:].real, s=0.1,
               # gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=rho_k_l[:,:].real, s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()

In [ ]:
x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=fft_w_[:,:].real, s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
pot_scr_z_ = fft_w_ * rho_k_l 

In [ ]:
x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot_scr_z_[:,:].real, s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (ttkw2.fft_xx[:,:,ttkw2.fft_nz//2]-0.5*ttkw2.fft_yy[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_yy[:,:,ttkw2.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.fft.ifftn(pot_scr_z_[:,:]).real, s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
y_plot = (ttkw2.fft_kyy[:,:,ttkw2.fft_nz//2]*2*np.sqrt(3)/3 +np.sqrt(3)/3* ttkw2.fft_kxx[:,:,ttkw2.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot_k_z[:,:,225//2].real, s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
plt.plot(np.fft.ifftn(pot_scr_z_)[0,:])

In [ ]:
np.shape(pot_r2)[2]

In [ ]:
np.stack((a1))

In [ ]:
a1 = np.array([[1,2],[3,4]])
a2 = np.array([[5,6],[7,8]])

In [ ]:
np.linspace(0.0001,0.08,10)

In [ ]:
import matplotlib.pyplot as plt

# 数据
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
temperatures = [22, 24, 19, 23, 25, 26, 24]

# 创建折线图
plt.figure(figsize=(10, 6))
plt.plot(days, temperatures, marker='o')  # 使用圆形标记每个数据点

# 添加标题和轴标签
plt.title("Temperature Variation Over a Week")
plt.xlabel("Day")
plt.ylabel("Temperature (°C)")

# 显示图表
plt.grid(True)
plt.show()